# 바깥 원판을 놓쳐도 허브 높이가 가까운 사례가 다른 선수에게도 있을까?

무릎 통과부터 접촉까지 직접 판독한 기존 검증 6명의 원본을 사용합니다. 분석용 입력은 640, 이전 b02 사례와 같은 조건은 800입니다. 같은 프레임을 두 번 평가하며 사람별 분할과 모델은 고정합니다.

In [ ]:
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from wlcv.config import ROOT
from wlcv.evaluation.hub_height_review import evaluate_heights, summarize_heights, plot_heights

reference = ROOT / "outputs/plate_resolution/2026-11-24"
output = ROOT / "outputs/validation_hub_height" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")

## 수동 판독을 적용하기 전의 선택 박스 복원

33번 노트북에서 영상 첫 장부터 시간 연결을 적용한 선택 박스를 불러옵니다. 같은 원본과 가중치로 각 판독 프레임의 후보를 다시 추론해 좌표, 신뢰도, 외곽 IoU를 확인합니다. 출력이 없는 프레임도 표에 남깁니다.

In [ ]:
frames, candidates = evaluate_heights(reference)
people = summarize_heights(frames)
output.mkdir(parents=True)
for name, table in [("frames", frames), ("candidates", candidates), ("people", people)]:
    table.to_csv(output / f"{name}.csv", index=False)
display(people.round(2))

## 이전 b02 사례와 같은 조건 찾기

실제 선택 박스가 외곽 IoU 0.50 미만이고, 같은 프레임에 IoU 0.50 이상인 후보와 수동 허브가 있을 때만 비교할 수 있습니다(`eligible`). 선택 박스의 세로 오차가 그 후보들 **모두**보다 작을 때 `pattern`으로 표시합니다. 두 조건의 오차가 같으면 반복 사례로 세지 않습니다. 같은 선수를 본 여러 프레임은 서로 연관되어 있습니다.

In [ ]:
columns = ["person_id", "frame_index", "imgsz", "source", "outer_iou", "matching_candidates",
           "height_error_px", "min_matching_error_px", "eligible", "pattern"]
display(frames.loc[frames.outer_iou.lt(.50) | frames.source.eq("missing"), columns].round(2))
display(people.groupby("imgsz")[["frames", "manual_hubs", "missing", "outer_hits", "eligible", "pattern"]].sum())

## 선수별로 외곽과 허브 높이를 함께 보기

점 하나는 원본 한 프레임에서 실제 선택된 박스입니다. 점이 오른쪽에 있을수록 바깥 테두리와 잘 겹치고, 아래에 있을수록 박스 중심 높이가 수동 허브에 가깝습니다. 두 입력 크기의 오차는 모두 해당 원본의 픽셀 단위입니다. 촬영 배율이 다른 선수의 px를 실제 거리로 비교할 수는 없습니다.

In [ ]:
fig = plot_heights(frames)
fig.savefig(output / "outer_iou_and_hub_height.png", dpi=150)
display(fig)
plt.close(fig)
print(output)